In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [8]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [9]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [10]:
df.show()

+------+-----------+--------+-------------------+-------+-------+
|amount|   category|   store|          timestamp|  tx_id|user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|    u48|
| 79.57|    książki|Warszawa|2026-04-12 08:05:43|TX00002|    u15|
|126.17|     odzież|Warszawa|2026-04-12 09:15:30|TX00003|    u18|
| 34.08|     odzież|Warszawa|2026-04-12 10:05:39|TX00004|    u10|
|428.88|    żywność|  Kraków|2026-04-12 09:04:36|TX00005|    u17|
|345.21|    książki|Warszawa|2026-04-12 09:36:31|TX00006|    u25|
|376.42|    żywność|Warszawa|2026-04-12 10:06:49|TX00007|    u15|
| 85.36|elektronika|  Gdańsk|2026-04-12 09:08:25|TX00008|    u24|
| 66.26|    żywność|  Kraków|2026-04-12 10:06:19|TX00009|    u05|
|660.41|     odzież|  Kraków|2026-04-12 08:29:24|TX00010|    u41|
|517.07|    żywność|  Kraków|2026-04-12 08:41:13|TX00011|    u21|
| 37.04|    żywność| Wrocław|2026-04-12 09:42:05|TX00012|    u14|
|591.09|  

In [11]:
df.toPandas().describe()

,amount,timestamp
count,10000.000000,10000
mean,401.154475,2026-04-12 09:24:21.459800320
min,5.000000,2026-04-12 08:00:06
25%,98.880000,2026-04-12 08:52:04
50%,210.080000,2026-04-12 09:22:57.500000
75%,449.745000,2026-04-12 09:55:22.249999872
max,9999.000000,2026-04-12 10:59:57
std,635.578181,NaN


In [12]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [ ]:
# zadanie 2.2

In [21]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum, round as _round


category_stats = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN")
    )
    .orderBy("category")
).show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



In [ ]:
# CZESC 3

In [13]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [14]:
hourly.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [ ]:
# zad 3.2

In [22]:
okno_30min= (
    df.groupBy(
        window("timestamp", "30 minutes"),
        "store"
    )
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy("window", "store")
)

(
    okno_30min
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN"
    )
    .show(truncate=False)
)

+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-

In [ ]:
# zad 3.3.

In [23]:
from pyspark.sql.functions import desc

krakow_hourly = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        count("tx_id").alias("liczba_tx")
    )
    .orderBy(desc("suma_PLN"))
)

(
    krakow_hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN"
    ).limit(1)
    .show(truncate=False)
)

+-------------------+-------------------+---------+---------+
|od                 |do                 |liczba_tx|suma_PLN |
+-------------------+-------------------+---------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|1169     |483309.86|
+-------------------+-------------------+---------+---------+



In [ ]:
# CZESC 4

In [15]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [ ]:
# zad 4.2

In [24]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: poniewaz okna sliding się na siebie nakładaja

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [25]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") grupuje transakcje jedynie po sklepie, a groupBy(window(...), "store")
# grupuje per okno i per sklep

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2

In [ ]:
### zad dom

In [17]:
from pyspark.sql.functions import window, avg, round as _round, asc

gdansk_info = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("avg_PLN"),
        count("tx_id").alias("liczba_tx")
    )
    .orderBy(asc("avg_PLN"))
)

gdansk_info.limit(1).show(truncate=False)

+------------------------------------------+-------+---------+
|window                                    |avg_PLN|liczba_tx|
+------------------------------------------+-------+---------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.01 |766      |
+------------------------------------------+-------+---------+



In [18]:
from pyspark.sql.functions import count

zad2 = (
    df.groupBy(
        window("timestamp", "30 minutes"),
        "category"
    )
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .filter(
        (col("window.start") == "2026-04-12 09:00:00")
    )
    .orderBy("category")
)

zad2.show(truncate=False)

+------------------------------------------+-----------+---------+
|window                                    |category   |liczba_tx|
+------------------------------------------+-----------+---------+
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|elektronika|611      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|książki    |622      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|odzież     |605      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|żywność    |567      |
+------------------------------------------+-----------+---------+



In [20]:
from pyspark.sql.functions import desc
zad3 = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy(desc("liczba_tx"))
)

zad3.limit(1).show(truncate=False)

+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |
+------------------------------------------+---------+

